## Setting up:

In [ ]:
import os, sys
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM
import folium

import pandas as pd
import warnings
import massbalancemachine as mbm
import pyproj
import matplotlib.pyplot as plt
import seaborn as sns
import xarray as xr
from cmcrameri import cm
from oggm import utils

from regions.RGI_01_02_US_Canada.scripts.config_US_CA import *

# from regions.RGI_11_Switzerland.scripts.oggm import initialize_oggm_glacier_directories, export_oggm_grids
from regions.RGI_11_Switzerland.scripts.glamos import merge_pmb_with_oggm_data, rename_stakes_by_elevation, check_point_ids_contain_glacier, remove_close_points, check_multiple_rgi_ids
from regions.RGI_11_French_Alps.scripts.glacioclim_preprocess import add_svf_from_rgi_zarr, plot_missing_svf_for_all_glaciers, add_svf_nearest_valid
from regions.TF_Europe.scripts.oggm import initialize_oggm_glacier_directories, export_oggm_grids

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

cfg = mbm.USCANADAConfig()

mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

## Pre-processing:

### Load WGMS data:

In [ ]:
df_stakes = pd.read_csv(
    os.path.join(cfg.dataPath, 'WGMS/raw/mass_balance_point.csv'))

df_RGIId = pd.read_csv(cfg.dataPath + 'WGMS/raw/glacier.csv')

# Create a mapping dictionary from id to rgi60_ids
id_to_rgi_map = dict(zip(df_RGIId['id'], df_RGIId['rgi60_ids']))

# Create a mapping dictionary from id to gtng_region
id_to_gtng_region_map = dict(zip(df_RGIId['id'], df_RGIId['gtng_region']))

# Add the RGIId column to the filtered DataFrame using glacier_id instead of id
df_stakes['RGIId'] = df_stakes['glacier_id'].map(id_to_rgi_map)
df_stakes['gtng_region'] = df_stakes['glacier_id'].map(id_to_gtng_region_map)

print('Unique countries in df_stakes:', df_stakes['country'].unique())
print('Unique gtng_regions in df_stakes:', df_stakes['gtng_region'].unique())

# Filter df_stakes to include only rows where country is US or CA
df_stakes_US_CA = df_stakes[df_stakes['country'].isin(
    ['US', 'CA'])].reset_index(drop=True)

df_stakes_US_CA

In [ ]:
# Create a mapping dictionary from gtng_region to all countries
country_to_gtng_region_map = df_RGIId.groupby(
    'gtng_region')['country'].unique().to_dict()
country_to_gtng_region_map

In [ ]:
import geopandas as gpd
# Load external glacier information from 01_Alaska.csv
df_stakes_add = pd.read_csv('01_Alaska.csv', encoding='latin1', sep=';')
# # Add the RGIId column to the filtered DataFrame using glacier_id instead of id
# df_stakes_add['RGIId'] = df_stakes_add['glacier_id'].map(id_to_rgi_map)
# df_stakes_add['gtng_region'] = df_stakes_add['glacier_id'].map(id_to_gtng_region_map)

# print('Unique countries in df_stakes_add:', df_stakes_add['country'].unique())
# print('Unique gtng_regions in df_stakes_add:', df_stakes_add['gtng_region'].unique())
df_stakes_add['glacier_name'].unique()

# Add RGI Ids through intersection with outlines:
glacier_outline_fname = os.path.join(
    cfg.dataPath, 'RGI_v6/RGI_01_Alaska/01_rgi60_Alaska.shp')

# check path exists
assert os.path.exists(
    glacier_outline_fname), f"Path {glacier_outline_fname} does not exist."
glacier_outline = gpd.read_file(glacier_outline_fname)

# rename columns
df_stakes_add.rename(columns={
    'latitude': 'POINT_LAT',
    'longitude': 'POINT_LON'
},
                     inplace=True)

# Add RGI IDs through intersection
df_stakes_add = mbm.data_processing.utils.get_rgi(
    data=df_stakes_add, glacier_outlines=glacier_outline)

# rename columns
df_stakes_add.rename(columns={
    'POINT_LAT': 'latitude',
    'POINT_LON': 'longitude'
},
                     inplace=True)

df_stakes_add['gtng_region'] = '01_alaska'

# Convert European d.m.y dates to YYYY-MM-DD format to match the other df
for col in ['begin_date', 'end_date']:
    df_stakes_add[col] = pd.to_datetime(df_stakes_add[col],
                                        dayfirst=True).dt.strftime('%Y-%m-%d')

df_stakes_add

In [ ]:
print(df_stakes_US_CA.gtng_region.unique())

# Filter to "non arctic" regions
df_stakes_US_CA = df_stakes_US_CA[df_stakes_US_CA['gtng_region'].isin(
    ['01_alaska', '02_western_canada_usa'])]

df_stakes_US_CA = pd.concat([df_stakes_US_CA, df_stakes_add],
                            ignore_index=True).reset_index(drop=True)

print('Number of measurements:', len(df_stakes_US_CA))
print('Number of glaciers:', len(df_stakes_US_CA.RGIId.unique()))

# Display glacier names with NaN RGIId
display(
    f"Number of rows with NaN RGIId: {df_stakes_US_CA['RGIId'].isna().sum()}")
display(
    df_stakes_US_CA[df_stakes_US_CA['RGIId'].isna()]['glacier_name'].unique())

# drop rows with NaN RGIId
df_stakes_US_CA = df_stakes_US_CA.dropna(subset=['RGIId']).reset_index(
    drop=True)

print('Number of measurements:', len(df_stakes_US_CA))
print('Number of glaciers:', len(df_stakes_US_CA.RGIId.unique()))

# Display glacier names with NaN RGIId
display(
    f"Number of rows with NaN RGIId: {df_stakes_US_CA['RGIId'].isna().sum()}")
display(
    df_stakes_US_CA[df_stakes_US_CA['RGIId'].isna()]['glacier_name'].unique())

# print number of glaciers per gtng_region
print(df_stakes_US_CA.groupby('gtng_region')['RGIId'].nunique())

In [ ]:
df_stakes_US_CA.head(2)

In [ ]:
# Select and rename columns
df_stakes_US_CA_renamed = df_stakes_US_CA.rename(
    columns={
        'point_id': 'POINT_ID',
        'latitude': 'POINT_LAT',
        'longitude': 'POINT_LON',
        'elevation': 'POINT_ELEVATION',
        'begin_date': 'FROM_DATE',
        'end_date': 'TO_DATE',
        'balance': 'POINT_BALANCE',
        'glacier_name': 'GLACIER',
        'year': 'YEAR',
        'country': 'COUNTRY',
        'balance_code': 'PERIOD',
        'gtng_region': "RGI_REGION"
    })

# Create new POINT_ID column
df_stakes_US_CA_renamed['POINT_ID'] = (
    df_stakes_US_CA_renamed['GLACIER'] + '_' +
    df_stakes_US_CA_renamed['YEAR'].astype(str) + '_' +
    df_stakes_US_CA['id'].astype(str) + '_' +
    df_stakes_US_CA_renamed['COUNTRY'])

# Only keep relevant columns in df
# df_stakes_US_CA_renamed = df_stakes_US_CA_renamed[[
#     'POINT_ID', 'POINT_LAT', 'POINT_LON', 'POINT_ELEVATION', 'FROM_DATE',
#     'TO_DATE', 'POINT_BALANCE', 'GLACIER', 'PERIOD', 'RGIId', 'YEAR',
#     'begin_date_unc', 'end_date_unc', "RGI_REGION"
# ]]

df_stakes_US_CA_renamed = df_stakes_US_CA_renamed[[
    'POINT_ID', 'POINT_LAT', 'POINT_LON', 'POINT_ELEVATION', 'FROM_DATE',
    'TO_DATE', 'POINT_BALANCE', 'GLACIER', 'PERIOD', 'RGIId', 'YEAR',
    "RGI_REGION"
]]

# Remove rows with NaN values in POINT_LAT, POINT_LON, and POINT_ELEVATION
df_stakes_US_CA_renamed = df_stakes_US_CA_renamed.dropna(
    subset=['POINT_LAT', 'POINT_LON', 'POINT_ELEVATION'])

# change date format to YYYYMMDD
df_stakes_US_CA_renamed['FROM_DATE'] = df_stakes_US_CA_renamed[
    'FROM_DATE'].astype(str).str.replace('-', '')
df_stakes_US_CA_renamed['TO_DATE'] = df_stakes_US_CA_renamed['TO_DATE'].astype(
    str).str.replace('-', '')

# Add data modification column to keep track of mannual changes
df_stakes_US_CA_renamed['DATA_MODIFICATION'] = ''

display(df_stakes_US_CA_renamed.head(2))

In [ ]:
# Check if any entry anywhere is NaN
display(df_stakes_US_CA_renamed[df_stakes_US_CA_renamed.isna().any(axis=1)])

In [ ]:
def plot_stakes_folium(
    df_pmb_clean,
    glacier_col="GLACIER",
    lat_col=None,
    lon_col=None,
    elev_col=None,
    id_col=None,
    center=None,
    zoom_start=10,
    color_map=None,
):
    """
    Create an interactive Folium map of stake points grouped by glacier.
    """

    # Infer column names if not provided
    if lat_col is None:
        lat_col = "lat" if "lat" in df_pmb_clean.columns else "POINT_LAT"
    if lon_col is None:
        lon_col = "lon" if "lon" in df_pmb_clean.columns else "POINT_LON"
    if elev_col is None:
        elev_col = "altitude" if "altitude" in df_pmb_clean.columns else "POINT_ELEVATION"
    if id_col is None:
        id_col = "stake_number" if "stake_number" in df_pmb_clean.columns else "POINT_ID"

    # Compute center if not provided
    if center is None:
        center_lat = float(df_pmb_clean[lat_col].median())
        center_lon = float(df_pmb_clean[lon_col].median())
    else:
        center_lat, center_lon = center

    m = folium.Map(location=[center_lat, center_lon], zoom_start=zoom_start)

    # Default colors (cycled) if user doesn't give explicit mapping
    default_colors = [
        "red", "blue", "green", "purple", "orange", "darkred", "cadetblue",
        "darkgreen", "darkpurple", "pink", "gray", "black"
    ]

    glaciers = sorted(df_pmb_clean[glacier_col].dropna().unique())

    if color_map is None:
        color_map = {
            g: default_colors[i % len(default_colors)]
            for i, g in enumerate(glaciers)
        }
    else:
        # fill missing glaciers with default cycling
        for i, g in enumerate(glaciers):
            color_map.setdefault(g, default_colors[i % len(default_colors)])

    # Add markers for each glacier
    for glacier_name, df_g in df_pmb_clean.groupby(glacier_col):
        if pd.isna(glacier_name):
            continue

        fg = folium.FeatureGroup(name=str(glacier_name))
        color = color_map[str(glacier_name)]

        for _, row in df_g.iterrows():
            stake_id = row.get(id_col, "NA")
            altitude = row.get(elev_col, "NA")

            folium.CircleMarker(
                location=[row[lat_col], row[lon_col]],
                radius=5,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.9,
                popup=f"{glacier_name} - Stake {stake_id}: {altitude} m",
            ).add_to(fg)

        fg.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)

    # Legend (auto-generated)
    legend_rows = "\n".join(
        f'<p><span style="color: {color_map[g]};">●</span> {g}</p>'
        for g in glaciers)

    legend_html = f"""
    <div style="
        position: fixed; bottom: 50px; left: 50px; z-index: 1000;
        background-color: white; padding: 10px; border-radius: 5px;
        border: 1px solid #999;
    ">
        <p><strong>Glaciers</strong></p>
        {legend_rows}
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

    return m


m = plot_stakes_folium(df_stakes_US_CA_renamed, color_map=None)
m

In [ ]:
# print max, min POINT_LON and POINT_LAT
print('Longitude range:', df_stakes_US_CA_renamed['POINT_LON'].min(), 'to',
      df_stakes_US_CA_renamed['POINT_LON'].max())

print('Latitude range:', df_stakes_US_CA_renamed['POINT_LAT'].min(), 'to',
      df_stakes_US_CA_renamed['POINT_LAT'].max())

In [ ]:
# save to csv
df_stakes_US_CA_renamed[['RGIId', 'RGI_REGION']].to_csv(
    os.path.join(cfg.dataPath, 'WGMS/raw/RGIids_stakes_US_CA.csv'))

### Add OGGM data:

In [ ]:
RGI_REGIONS = ["01", "02"]  # Alaska & Western Canada & USA

# Mapping from RGI number to gtng_region string in df_stakes
RGI_TO_GTNG = {
    "01": "01_alaska",
    "02": "02_western_canada_usa",
}

# Run for each region separately to avoid memory issues, and create a dictionary of RGIIds for each region
rgi_ids_list_rgn = {
    '01':
    df_stakes_US_CA_renamed[df_stakes_US_CA_renamed['RGI_REGION'] ==
                            '01_alaska']['RGIId'].unique().tolist(),
    '02':
    df_stakes_US_CA_renamed[df_stakes_US_CA_renamed['RGI_REGION'] ==
                            '02_western_canada_usa']
    ['RGIId'].unique().tolist()
}

INITIALIZE_OGGM = True
if INITIALIZE_OGGM:
    df_stakes_topo_list = []

    for RGI_REGION in RGI_REGIONS:
        # Initialize OGGM glacier directories (needs just the number)
        rgi_ids_list = rgi_ids_list_rgn[RGI_REGION]
        print(
            f"[{RGI_REGION}] Initializing OGGM glacier directories for {len(rgi_ids_list)} glaciers..."
        )
        gdirs, rgi_ids_list = initialize_oggm_glacier_directories(
            cfg,
            rgi_region=RGI_REGION,
            rgi_version="62",
            base_url=
            "https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/L1-L2_files/2025.6/elev_bands_w_data/",
            log_level='WARNING',
            task_list=None,
            rgi_ids_list=rgi_ids_list)
        export_oggm_grids(cfg,
                          gdirs,
                          rgi_region=RGI_REGION,
                          subset_rgis=rgi_ids_list)

        # Filter stakes using the full gtng_region string
        gtng_key = RGI_TO_GTNG[RGI_REGION]
        df_stakes_region = df_stakes_US_CA_renamed[
            df_stakes_US_CA_renamed['RGI_REGION'] == gtng_key]
        print(
            f"[{RGI_REGION}] {len(df_stakes_region)} stake measurements for {gtng_key}"
        )

        df_stakes_topo_ = merge_pmb_with_oggm_data(
            df_pmb=df_stakes_region,
            gdirs=gdirs,
            rgi_region=RGI_REGION,
            rgi_version="62",
        )
        df_stakes_topo_list.append(df_stakes_topo_)

    # Concatenate all regions
    df_stakes_topo = pd.concat(df_stakes_topo_list, ignore_index=True)

# Restrict to within glacier shape
df_stakes_topo = df_stakes_topo[df_stakes_topo['within_glacier_shape'] == True]
df_stakes_topo = df_stakes_topo.drop(columns=['within_glacier_shape'])

# Display rows that have any NaN values
display(df_stakes_topo[df_stakes_topo.isna().any(axis=1)])

### Merge close stakes:

In [ ]:
from tqdm import tqdm

df_pmb_topo = pd.DataFrame()
for gl in tqdm(df_stakes_topo.GLACIER.unique(), desc='Merging stakes'):
    print(f'-- {gl.capitalize()}:')
    df_gl = df_stakes_topo[df_stakes_topo.GLACIER == gl]
    df_gl_cleaned = remove_close_points(df_gl)
    df_pmb_topo = pd.concat([df_pmb_topo, df_gl_cleaned])
df_pmb_topo.drop(['x', 'y'], axis=1, inplace=True)
df_pmb_topo.reset_index(inplace=True, drop=True)

### Add skyview factor:

In [ ]:
# Example of one svf file
rgi_id = df_pmb_topo.loc[0].RGIId

rgi_gl = "RGI60-01.00037"

# read ds with svf
path_masked_xr = os.path.join(cfg.dataPath,
                              'RGI_v6/RGI_01_Alaska/xr_masked_grids/')

xr.open_zarr(path_masked_xr + f'{rgi_gl}.zarr').svf.plot()

In [ ]:
RGI_REGIONS_CONFIG = {
    "01":
    os.path.join(cfg.dataPath, 'RGI_v6/RGI_01_Alaska/xr_masked_grids/'),
    "02":
    os.path.join(cfg.dataPath,
                 'RGI_v6/RGI_02_WesternCanadaUS/xr_masked_grids/'),
}

df_pmb_topo_svf_list = []

for rgi_id, path_masked_xr in RGI_REGIONS_CONFIG.items():
    # Filter to current region
    gtng_key = RGI_TO_GTNG[rgi_id]
    df_pmb_topo_region = df_pmb_topo[df_pmb_topo['RGI_REGION'] == gtng_key]
    print(
        f"[{rgi_id}] Processing {len(df_pmb_topo_region)} points from {gtng_key}"
    )

    df_pmb_topo_svf_ = add_svf_from_rgi_zarr(
        df_pmb_topo_region,
        path_masked_xr,
        rgi_col="RGIId",
        lon_col="POINT_LON",
        lat_col="POINT_LAT",
        svf_var="svf",
        out_col="svf",
    )
    df_pmb_topo_svf_list.append(df_pmb_topo_svf_)

# Concatenate all regions
df_pmb_topo_svf = pd.concat(df_pmb_topo_svf_list, ignore_index=True)

df_missing = df_pmb_topo_svf[df_pmb_topo_svf["svf"].isna()].copy()
print("Missing SVF points:", len(df_missing))
print("Glaciers affected:", sorted(df_missing["RGIId"].unique()))

In [ ]:
plot_missing_svf_for_all_glaciers(
    df_with_svf=df_pmb_topo_svf,
    path_masked_xr=path_masked_xr,
    plot_valid_points=True,
    save_dir=
    None  # or e.g. os.path.join(cfg.dataPath, "diagnostics/svf_missing")
)

In [ ]:
paths_masked_xr = {
    "01_alaska":
    "/scratch-3/vmarijn/MassBalanceMachine/../data/RGI_v6/RGI_01_Alaska/xr_masked_grids/",
    "02_western_canada_usa":
    "/scratch-3/vmarijn/MassBalanceMachine/../data/RGI_v6/RGI_02_WesternCanadaUS/xr_masked_grids/",
}

dfs_out = []

for region, df_region in df_pmb_topo.groupby("RGI_REGION", sort=False):
    if region not in paths_masked_xr:
        raise ValueError(f"No masked xr path configured for region {region}")

    path_masked_xr = paths_masked_xr[region]

    print(f"\nProcessing region: {region}")
    print(f"Rows: {len(df_region)}")

    df_region_svf = add_svf_nearest_valid(
        df_region.copy(),
        path_masked_xr,
        rgi_col="RGIId",
        lon_col="POINT_LON",
        lat_col="POINT_LAT",
        svf_var="svf",
        out_col="svf",
        max_radius=60,
    )

    dfs_out.append(df_region_svf)

df_pmb_topo_svf_new = pd.concat(dfs_out, ignore_index=True)

assert (len(df_pmb_topo) == len(df_pmb_topo_svf_new))
print("Missing SVF points after nearest-valid fill:",
      df_pmb_topo_svf_new["svf"].isna().sum())

### Give new stake ids:

In [ ]:
df_pmb_new_ids = rename_stakes_by_elevation(df_pmb_topo_svf_new)

# Check the condition
check_point_ids_contain_glacier(df_pmb_new_ids)

print('Number of winter and annual samples:', len(df_pmb_new_ids))
print('Number of annual samples:',
      len(df_pmb_new_ids[df_pmb_new_ids.PERIOD == 'annual']))
print('Number of winter samples:',
      len(df_pmb_new_ids[df_pmb_new_ids.PERIOD == 'winter']))

# Histogram of mass balance
df_pmb_new_ids['POINT_BALANCE'].hist(bins=20)
plt.xlabel('Mass balance [m w.e.]')

### Final cleaning:

In [ ]:
import re
import numpy as np


def clean_date(val):
    """Coerce a raw date value to a clean YYYYMMDD string, or NaN if unrecoverable."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return np.nan
    s = re.sub(r'\.0*$', '', str(val).strip())  # Remove trailing .0
    s = re.sub(r'\D', '', s)  # Keep digits only
    return s if len(s) == 8 else np.nan


df_pmb_clean = df_pmb_new_ids.copy()

# Clean dates
df_pmb_clean["FROM_DATE"] = df_pmb_clean["FROM_DATE"].apply(clean_date)
df_pmb_clean["TO_DATE"] = df_pmb_clean["TO_DATE"].apply(clean_date)

# Report and remove unrecoverable rows
bad_mask = df_pmb_clean["FROM_DATE"].isna() | df_pmb_clean["TO_DATE"].isna()
if bad_mask.any():
    print(f"⚠️  Dropping {bad_mask.sum()} rows with unrecoverable dates:")
    print(df_pmb_new_ids.loc[bad_mask, ["FROM_DATE", "TO_DATE"]].to_string())
    df_pmb_clean = df_pmb_clean[~bad_mask].reset_index(drop=True)

# Extract months
df_pmb_clean["MONTH_START"] = df_pmb_clean["FROM_DATE"].str[4:6]
df_pmb_clean["MONTH_END"] = df_pmb_clean["TO_DATE"].str[4:6]


def print_months(df, label):
    winter = df[df.PERIOD == "winter"]
    annual = df[df.PERIOD == "annual"]
    print(f"\n{label}")
    print("Winter measurement months:")
    print("  Unique start months:", sorted(winter["MONTH_START"].unique()))
    print("  Unique end months:  ", sorted(winter["MONTH_END"].unique()))
    print("\nAnnual measurement months:")
    print("  Unique start months:", sorted(annual["MONTH_START"].unique()))
    print("  Unique end months:  ", sorted(annual["MONTH_END"].unique()))


# --- Before filtering ---
print_months(df_pmb_clean, "Before filtering")

# Normalise PERIOD once
period = df_pmb_clean["PERIOD"].astype(str).str.strip().str.lower()

In [ ]:
# Compute duration
df_pmb_clean["FROM_DATE_dt"] = pd.to_datetime(df_pmb_clean["FROM_DATE"],
                                              format="%Y%m%d",
                                              errors="coerce")
df_pmb_clean["TO_DATE_dt"] = pd.to_datetime(df_pmb_clean["TO_DATE"],
                                            format="%Y%m%d",
                                            errors="coerce")
df_pmb_clean["duration_days"] = (df_pmb_clean["TO_DATE_dt"] -
                                 df_pmb_clean["FROM_DATE_dt"]).dt.days


def print_months(df, label):
    winter = df[df.PERIOD == "winter"]
    annual = df[df.PERIOD == "annual"]
    print(f"\n{label}")
    print("Winter measurement months:")
    print("  Unique start months:", sorted(winter["MONTH_START"].unique()))
    print("  Unique end months:  ", sorted(winter["MONTH_END"].unique()))
    print("\nAnnual measurement months:")
    print("  Unique start months:", sorted(annual["MONTH_START"].unique()))
    print("  Unique end months:  ", sorted(annual["MONTH_END"].unique()))


def print_combos(df, period, label):
    sub = df[df["PERIOD"] == period].copy()
    combo_counts = (sub.groupby(["MONTH_START", "MONTH_END"]).agg(
        count=("duration_days", "size"),
        mean_days=("duration_days", "mean"),
        min_days=("duration_days", "min"),
        max_days=("duration_days", "max")).reset_index().sort_values(
            ["MONTH_START", "MONTH_END"]))
    combo_counts["mean_days"] = combo_counts["mean_days"].round(0).astype(int)
    print(f"\n{label}")
    print(combo_counts.to_string(index=False))


# --- Before anything ---
print_months(df_pmb_clean, "Before any changes")
print_combos(df_pmb_clean, "annual",
             "Annual: all combinations before shifting")
print_combos(df_pmb_clean, "winter",
             "Winter: all combinations before shifting")

# ======================
# 1) SHIFT JULY STARTS TO AUGUST 1st (before any filtering)
# ======================
for period_type in ["annual", "winter"]:
    mask_july = ((df_pmb_clean["PERIOD"] == period_type) &
                 (df_pmb_clean["MONTH_START"] == "07"))
    n_july = int(mask_july.sum())
    df_pmb_clean.loc[mask_july, "FROM_DATE"] = (
        df_pmb_clean.loc[mask_july, "FROM_DATE_dt"].apply(
            lambda d: d.replace(month=8, day=1).strftime("%Y%m%d")))
    # Recompute derived columns
    df_pmb_clean.loc[mask_july, "FROM_DATE_dt"] = pd.to_datetime(
        df_pmb_clean.loc[mask_july, "FROM_DATE"], format="%Y%m%d")
    df_pmb_clean.loc[mask_july, "duration_days"] = (
        df_pmb_clean.loc[mask_july, "TO_DATE_dt"] -
        df_pmb_clean.loc[mask_july, "FROM_DATE_dt"]).dt.days
    df_pmb_clean.loc[mask_july, "MONTH_START"] = "08"
    print(
        f"Shifted {n_july} {period_type} measurements from July to August 1st start"
    )

print_months(df_pmb_clean, "After shifting July starts to August")

# ======================
# 2) FILTER INVALID ANNUAL
# ======================
# valid_annual_end = ["04", "05", "06", "07", "08", "09", "10"]
# valid_annual_start = [
#     "08", "09", "10", "11", "12", "01", "02", "03", "04", "05"
# ]

valid_annual_end = ["05", "06", "07", "08", "09", "10"]
valid_annual_start = [
    "08", "09", "10", "11", "12", 
]

mask_invalid_annual = ((df_pmb_clean["PERIOD"] == "annual") & ~(
    (df_pmb_clean["MONTH_END"].isin(valid_annual_end)) &
    (df_pmb_clean["MONTH_START"].isin(valid_annual_start)) &
    (df_pmb_clean["duration_days"] <= 460)))
n_invalid_annual = int(mask_invalid_annual.sum())
print(f"\nInvalid annual rows to remove: {n_invalid_annual}")
print("\nAnnual combinations being removed:")
print(df_pmb_clean[mask_invalid_annual].groupby([
    "MONTH_START", "MONTH_END"
]).agg(count=("duration_days", "size"),
       mean_days=("duration_days", "mean"),
       min_days=("duration_days", "min"),
       max_days=("duration_days", "max")).reset_index().to_string(index=False))
df_pmb_clean = df_pmb_clean.loc[~mask_invalid_annual].copy()

# ======================
# 3) FILTER INVALID WINTER
# ======================
valid_winter_end = ["01", "02", "03", "04", "05", "06", "07"]
valid_winter_start = ["08", "09", "10", "11", "12"]

mask_invalid_winter = ((df_pmb_clean["PERIOD"] == "winter") & ~(
    (df_pmb_clean["MONTH_END"].isin(valid_winter_end)) &
    (df_pmb_clean["MONTH_START"].isin(valid_winter_start)) &
    (df_pmb_clean["duration_days"] <= 430)))
n_invalid_winter = int(mask_invalid_winter.sum())
print(f"\nInvalid winter rows to remove: {n_invalid_winter}")
print("\nWinter combinations being removed:")
print(df_pmb_clean[mask_invalid_winter].groupby([
    "MONTH_START", "MONTH_END"
]).agg(count=("duration_days", "size"),
       mean_days=("duration_days", "mean"),
       min_days=("duration_days", "min"),
       max_days=("duration_days", "max")).reset_index().to_string(index=False))
df_pmb_clean = df_pmb_clean.loc[~mask_invalid_winter].copy()

print_months(df_pmb_clean, "After all filtering")

In [ ]:
columns_to_drop = [
    'begin_date_unc', 'end_date_unc', 'RGI_REGION', 'MONTH_START', 'MONTH_END',
    'FROM_DATE_dt', 'TO_DATE_dt', 'duration_days', 'DATA_MODIFICATION'
]

df_mb_alaska = df_pmb_clean[df_pmb_clean.RGI_REGION == '01_alaska'].drop(
    columns=columns_to_drop, errors='ignore')
df_mb_alaska.to_csv(os.path.join(cfg.dataPath, path_PMB_WGMS_csv,
                                 'ALA_wgms_dataset_all.csv'),
                    index=False)

df_mb_caw = df_pmb_clean[df_pmb_clean.RGI_REGION ==
                         '02_western_canada_usa'].drop(columns=columns_to_drop,
                                                       errors='ignore')
df_mb_caw.to_csv(os.path.join(cfg.dataPath, path_PMB_WGMS_csv,
                              'CAW_wgms_dataset_all.csv'),
                 index=False)

# Histogram of mass balance
df_pmb_clean['POINT_BALANCE'].hist(bins=20)
plt.xlabel('Mass balance [m w.e.]')

In [ ]:
import folium


def plot_stakes_folium(
    df_pmb_clean,
    glacier_col="GLACIER",
    lat_col=None,
    lon_col=None,
    elev_col=None,
    id_col=None,
    center=None,
    zoom_start=10,
    color_map=None,
):
    """
    Create an interactive Folium map of stake points grouped by glacier.
    """

    # Infer column names if not provided
    if lat_col is None:
        lat_col = "lat" if "lat" in df_pmb_clean.columns else "POINT_LAT"
    if lon_col is None:
        lon_col = "lon" if "lon" in df_pmb_clean.columns else "POINT_LON"
    if elev_col is None:
        elev_col = "altitude" if "altitude" in df_pmb_clean.columns else "POINT_ELEVATION"
    if id_col is None:
        id_col = "stake_number" if "stake_number" in df_pmb_clean.columns else "POINT_ID"

    # Compute center if not provided
    if center is None:
        center_lat = float(df_pmb_clean[lat_col].median())
        center_lon = float(df_pmb_clean[lon_col].median())
    else:
        center_lat, center_lon = center

    m = folium.Map(location=[center_lat, center_lon], zoom_start=zoom_start)

    # Default colors (cycled) if user doesn't give explicit mapping
    default_colors = [
        "red", "blue", "green", "purple", "orange", "darkred", "cadetblue",
        "darkgreen", "darkpurple", "pink", "gray", "black"
    ]

    glaciers = sorted(df_pmb_clean[glacier_col].dropna().unique())

    if color_map is None:
        color_map = {
            g: default_colors[i % len(default_colors)]
            for i, g in enumerate(glaciers)
        }
    else:
        # fill missing glaciers with default cycling
        for i, g in enumerate(glaciers):
            color_map.setdefault(g, default_colors[i % len(default_colors)])

    # Add markers for each glacier
    for glacier_name, df_g in df_pmb_clean.groupby(glacier_col):
        if pd.isna(glacier_name):
            continue

        fg = folium.FeatureGroup(name=str(glacier_name))
        color = color_map[str(glacier_name)]

        for _, row in df_g.iterrows():
            stake_id = row.get(id_col, "NA")
            altitude = row.get(elev_col, "NA")

            folium.CircleMarker(
                location=[row[lat_col], row[lon_col]],
                radius=5,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.9,
                popup=f"{glacier_name} - Stake {stake_id}: {altitude} m",
            ).add_to(fg)

        fg.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)

    # Legend (auto-generated)
    legend_rows = "\n".join(
        f'<p><span style="color: {color_map[g]};">●</span> {g}</p>'
        for g in glaciers)

    legend_html = f"""
    <div style="
        position: fixed; bottom: 50px; left: 50px; z-index: 1000;
        background-color: white; padding: 10px; border-radius: 5px;
        border: 1px solid #999;
    ">
        <p><strong>Glaciers</strong></p>
        {legend_rows}
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

    return m


m = plot_stakes_folium(df_pmb_clean, color_map=None)
m

In [ ]:
print('Number of winter and annual samples:', len(df_pmb_clean))
print('Number of annual samples:',
      len(df_pmb_clean[df_pmb_clean.PERIOD == 'annual']))
print('Number of winter samples:',
      len(df_pmb_clean[df_pmb_clean.PERIOD == 'winter']))

# Number of measurements per year:
fig, axs = plt.subplots(2, 1, figsize=(20, 15))
ax = axs.flatten()[0]
df_pmb_clean.groupby(['YEAR', 'PERIOD']).size().unstack().plot(
    kind='bar',
    stacked=True,
    color=[mbm.plots.COLOR_ANNUAL, mbm.plots.COLOR_WINTER],
    ax=ax)
ax.set_title('Number of measurements per year for all glaciers')

ax = axs.flatten()[1]
num_gl = df_pmb_clean.groupby(['GLACIER']).size().sort_values()
num_gl.plot(kind='bar', ax=ax)
ax.set_title('Number of total measurements per glacier since 1951')
plt.tight_layout()

In [ ]:
# print max, min POINT_LON and POINT_LAT
stakes_alaska = df_pmb_clean[df_pmb_clean.RGI_REGION=='01_alaska']
print('Longitude range:', stakes_alaska['POINT_LON'].min(), 'to',
      stakes_alaska['POINT_LON'].max())

print('Latitude range:', stakes_alaska['POINT_LAT'].min(), 'to',
      stakes_alaska['POINT_LAT'].max())

stakes_caw = df_pmb_clean[df_pmb_clean.RGI_REGION=='02_western_canada_usa']
print('Longitude range:', stakes_caw['POINT_LON'].min(), 'to',
      stakes_caw['POINT_LON'].max())

print('Latitude range:', stakes_caw['POINT_LAT'].min(), 'to',
      stakes_caw['POINT_LAT'].max())